# Hartree–Fock for $\mathrm{H_2}$ in a minimal STO-3G basis

*physchem · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/hartree_fock_h2)

The textbook self-consistent field calculation of Szabo and Ostlund, written with tensors. Each hydrogen carries one 1s function made of three Gaussians (STO-3G); overlap, kinetic, nuclear-attraction and two-electron integrals have closed forms, the Fock matrix is a contraction of the density with the $(\mu\nu|\lambda\sigma)$ tensor, and the Roothaan equations $FC = SC\varepsilon$ are solved with eigh for a whole batch of bond lengths at once. At $R = 1.4$ bohr it gives $E = -1.1167$ hartree; the scan finds the bond length and shows why restricted Hartree–Fock cannot break the bond.

**Vector form**

$$F(P)\,C = S\,C\,\varepsilon,\qquad F = H^{\text{core}} + J(P) - \tfrac12 K(P),\qquad P = 2\,C_{\text{occ}}C_{\text{occ}}^{\mathsf T},\qquad E = \tfrac12\operatorname{tr}\big[P\,(H^{\text{core}} + F)\big] + \frac{Z_AZ_B}{R}$$

**Index (tensor) form**

$$S_{\mu\nu} = c_{\mu p}\,c_{\nu q}\,S_{\mu\nu pq},\qquad F_{\mu\nu} = H_{\mu\nu} + P_{\lambda\sigma}\big[(\mu\nu|\lambda\sigma) - \tfrac12(\mu\lambda|\nu\sigma)\big],\qquad X^{\mathsf T}_{\mu\kappa}F_{\kappa\rho}X_{\rho\nu}\,C^{\prime}_{\nu a} = \varepsilon_a\,C^{\prime}_{\mu a},\qquad P_{\lambda\sigma} = 2\,C_{\lambda a}C_{\sigma a}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\alpha_{\mu p},\ c_{\mu p}$ | `sto3g(zeta)` | (M, K) | three Gaussian exponents per function, $\alpha_p\zeta^2$, and contraction coefficients with the primitive normalisation $(2\alpha/\pi)^{3/4}$ folded in |
| $A_{b\mu a}$ | `geometry(R)` | (B, 2, 3) | the two nuclei (and basis centres) for every bond length $R_b$ in the batch |
| $S_{\mu\nu pq}$ | `(pi/zeta)**1.5 * exp(-mu*AB2)` | (B, M, M, K, K) | two Gaussians multiply into one Gaussian on $P = (\alpha A + \beta B)/(\alpha + \beta)$ |
| $S_{\mu\nu} = c_{\mu p}c_{\nu q}S_{\mu\nu pq}$ | `einsum('mp,nq,bmnpq->bmn', c, c, prim)` | (B, M, M) | repeated $p, q$ ⇒ summed: contraction over primitives; the same for $T$, $V$ |
| $V_{\mu\nu}$ | `nuclear_attraction(pairs, c, nuclei, Z)` | (B, M, M) | $-\sum_c Z_c\,\tfrac{2\pi}{\alpha+\beta}\,e^{-\dots}F_0\big((\alpha+\beta)/P - C/^2\big)$ with the Boys function $F_0(t) = \tfrac12\sqrt{\pi/t}\,\mathrm{erf}\sqrt t$ |
| $(\mu\nu\vert \lambda\sigma)$ | `electron_repulsion(pairs, c)` | (B, M, M, M, M) | bra and ket pair tensors broadcast against each other, then $p, q, r, t$ summed in one einsum |
| $X = S^{-1/2}$ | `U @ diag(s**-0.5) @ U.T` | (B, M, M) | symmetric (Löwdin) orthogonalisation: $X^{\mathsf T}SX = 1$ |
| $P_{\lambda\sigma}(\mu\nu\vert \lambda\sigma)$ | `einsum('bls,bmnls->bmn', P, eri)` | (B, M, M) | Coulomb matrix $J$; repeated $\lambda, \sigma$ ⇒ summed |
| $P_{\lambda\sigma}(\mu\lambda\vert \nu\sigma)$ | `einsum('bls,bmlns->bmn', P, eri)` | (B, M, M) | exchange matrix $K$: the same tensor read with the indices swapped |
| $\varepsilon_a,\ C^{\prime}_{\nu a}$ | `torch.linalg.eigh(X.mT @ F @ X)` | (B, M), (B, M, M) | one batched eigh per SCF pass; $C = XC^{\prime}$ |
| $N = P_{\mu\nu}S_{\nu\mu}$ | `einsum('bmn,bnm->b', P, S)` | (B,) | Mulliken electron count: exactly 2 |
| $E_b$ | `0.5*einsum("bmn,bmn->b", P, H + F) + Z_A*Z_B/R` | (B,) | the potential curve, one entry per bond length |

Atomic units: lengths in bohr ($a_0 = 0.529$ Å) and energies in hartree ($E_h = 27.21$ eV). The basis is STO-3G with Slater exponents $\zeta_{\mathrm H} = 1.24$ and $\zeta_{\mathrm{He}} = 2.0925$ as in Szabo and Ostlund, Modern Quantum Chemistry (1982; Dover edition 1996), chapter 3; with two basis functions every matrix is $2\times2$, so you can check each number against the book: $S_{12} = 0.6593$, $T_{11} = 0.7600$, $E = -1.1167$ hartree at $R = 1.4$ bohr, and $-2.8607$ hartree for $\mathrm{HeH^+}$ at $R = 1.4632$ bohr. The SCF starts from $P = 0$ (the core Hamiltonian); for $\mathrm{H_2}$ symmetry fixes the orbitals $\psi_{1,2} \propto \phi_A \pm \phi_B$ and it converges at once, while $\mathrm{HeH^+}$ needs a dozen passes as charge flows onto helium. Every bond length is a batch entry, so the SCF runs for the whole curve in one loop. Restricted Hartree–Fock puts both electrons in the same spatial orbital $\psi_1$; at large $R$ that state is half $\mathrm{H + H}$ and half the ionic $\mathrm{H^+ + H^-}$, so the curve rises far above $2E_{\mathrm H}$. The dashed curve is the full configuration interaction in the same basis (the $3\times3$ singlet matrix of $|1^2\rangle$, $|1^12^1\rangle$, $|2^2\rangle$): it mixes in $|2^2\rangle$, dissociates correctly, and its gap to RHF at equilibrium (about $-0.02$ hartree) is the correlation energy. A minimal basis is far from the exact $\mathrm{H_2}$ energy, $-1.1745$ hartree at $R = 1.4$ bohr; larger bases need p and d functions and the general Gaussian integral recursions. Try $\mathrm{HeH^+}$: there the RHF curve does reach the right limit, $\mathrm{He + H^+}$, because both electrons end up on helium.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/hartree_fock_h2/main.py
"""Main loop: STO-3G basis → integrals for a batch of bond lengths → SCF → potential curve and orbitals."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import basis_on_points, nuclear_repulsion

# Szabo & Ostlund's worked examples (STO-3G, ζ_H = 1.24, ζ_He = 2.0925): (R in bohr, total energy in hartree)
REFERENCE = {"H2": (1.4, -1.1167), "HeH+": (1.4632, -2.8607)}


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    name = params["molecule"]
    Z, zeta, n_atom = tensors.molecule(name, dev)
    alpha, coef = tensors.sto3g(zeta)  # (M, K) each
    n_R = 24 if fast_mode() else params["n_bond_lengths"]
    R_ref, E_ref = REFERENCE[name]
    # the scan plus the reference geometry as one extra batch entry
    R = torch.cat([tensors.bond_lengths(params["R_min"], params["R_max"], n_R, dev), torch.tensor([R_ref], device=dev)])
    nuclei = tensors.geometry(R)  # (B, 2, 3)
    ints = solve.integrals(alpha, coef, nuclei, Z)
    print(f"{name} in STO-3G: ζ = {zeta.tolist()}, exponents α_μp (bohr⁻²):")
    for m in range(len(zeta)):
        print(f"  atom {m}: α = {[round(a, 5) for a in alpha[m].tolist()]}")
    print(f"{len(R)} bond lengths at once: S, T, V {tuple(ints['S'].shape)}, (μν|λσ) {tuple(ints['eri'].shape)}")

    res = solve.scf(ints, tensors.N_ELECTRONS)
    E_nuc = nuclear_repulsion(nuclei, Z)
    E = res["E_el"] + E_nuc  # (B,) total energy
    E_ci = solve.full_ci(ints, res["C"]) + E_nuc
    n_elec = torch.einsum("bmn,bnm->b", res["P"], ints["S"])  # tr(PS)
    print(
        f"SCF converged in {res['n_iter']} iterations (max |ΔP| = {res['history'][-1]:.1e}); tr(PS) = "
        f"{n_elec.min().item():.6f} … {n_elec.max().item():.6f}"
    )

    k = len(R) - 1  # the reference geometry
    S, H, F, C = ints["S"][k], ints["H"][k], res["F"][k], res["C"][k]
    print(f"\nR = {R_ref} bohr:")
    print(f"  S = [[{S[0, 0]:.4f}, {S[0, 1]:.4f}], [{S[1, 0]:.4f}, {S[1, 1]:.4f}]]")
    print(f"  H_core = [[{H[0, 0]:.4f}, {H[0, 1]:.4f}], [{H[1, 0]:.4f}, {H[1, 1]:.4f}]]")
    print(f"  F = [[{F[0, 0]:.4f}, {F[0, 1]:.4f}], [{F[1, 0]:.4f}, {F[1, 1]:.4f}]]")
    print(f"  orbital energies ε = {res['eps'][k, 0]:.4f}, {res['eps'][k, 1]:.4f} hartree")
    print(f"  MO coefficients C (columns = MOs): [[{C[0, 0]:.4f}, {C[0, 1]:.4f}], [{C[1, 0]:.4f}, {C[1, 1]:.4f}]]")
    print(f"  E_electronic = {res['E_el'][k]:.6f}, E_nuclear = {E_nuc[k]:.6f}")
    print(f"  E_RHF = {E[k]:.6f} hartree   (Szabo & Ostlund: {E_ref})   full CI = {E_ci[k]:.6f}")

    R_scan, E_scan, Eci_scan = R[:-1], E[:-1], E_ci[:-1]
    r0, e0 = solve.equilibrium(R_scan, E_scan)
    r0_ci, e0_ci = solve.equilibrium(R_scan, Eci_scan)
    E_atoms = solve.separated_atoms(alpha, coef, Z, n_atom)
    print(f"\nequilibrium (RHF):     R_e = {r0:.4f} bohr = {0.529177 * r0:.4f} Å,  E = {e0:.6f} hartree")
    print(f"equilibrium (full CI): R_e = {r0_ci:.4f} bohr,  E = {e0_ci:.6f} hartree")
    print(f"separated atoms ({'H + H' if name == 'H2' else 'He + H⁺'}): {E_atoms:.6f} hartree")
    print(
        f"at R = {R_scan[-1]:.1f} bohr: RHF {E_scan[-1]:.6f}, full CI {Eci_scan[-1]:.6f}"
        f"  → RHF misses the limit by {E_scan[-1].item() - E_atoms:.4f} hartree"
    )

    z = tensors.axis_points(-3.0, params["R_max"] + 3.0, 300, dev)
    phi = basis_on_points(alpha, coef, nuclei[:-1], z)  # (B, Z, M)
    psi = torch.einsum("bzm,bma->bza", phi, res["C"][:-1])  # MOs along the axis
    return plot.animate(
        name, R_scan, E_scan, Eci_scan, E_atoms, res["eps"][:-1], res["C"][:-1], z[:, 2], psi, (r0, e0),
        out_dir / "hartree_fock_h2.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/hartree_fock_h2/equation.py
r"""Physics → tensor translation: integrals over s-type Gaussians and the Roothaan–Hall Fock matrix.

Indices: b bond length, μ ν λ σ (m n l s) contracted basis functions, p q r t primitives, c nucleus, a Cartesian axis.

Two primitives with exponents α, β on centres A, B multiply into one Gaussian on P = (αA + βB)/(α + β):
    S_pq = (π/(α+β))^{3/2} exp(-αβ/(α+β) |A-B|²)
    T_pq = αβ/(α+β) (3 - 2αβ/(α+β) |A-B|²) S_pq
    V_pq = -Σ_c Z_c 2π/(α+β) exp(-αβ/(α+β) |A-B|²) F₀((α+β)|P - C_c|²)                       (c summed)
    (pq|rt) = 2π^{5/2} / (ζ η √(ζ+η)) e^{-…AB} e^{-…CD} F₀(ζη/(ζ+η) |P - Q|²),   ζ = α+β, η = γ+δ
and every contracted integral is a sum over primitives, e.g. S_μν = c_μp c_νq S_μνpq           (p, q summed)

    F_μν = H_μν + P_λσ [(μν|λσ) - ½ (μλ|νσ)]                                                  (λ, σ summed)
    P_λσ = 2 C_λa C_σa                                                                      (a over occupied MOs)
    E = ½ P_μν (H_μν + F_μν) + Z_A Z_B / R
"""

import math

import torch


def boys_f0(t: torch.Tensor) -> torch.Tensor:
    """F₀(t) = ∫₀¹ e^{-t u²} du = ½ √(π/t) erf(√t), with the series 1 - t/3 near t = 0."""
    ts = t.clamp_min(1e-12)
    return torch.where(t < 1e-8, 1 - t / 3, 0.5 * torch.sqrt(math.pi / ts) * torch.erf(torch.sqrt(ts)))


def gaussian_pairs(alpha: torch.Tensor, centres: torch.Tensor) -> dict:
    """Products of every primitive pair (μp, νq).   alpha (M, K), centres (B, M, 3).

    Returns zeta = α+β (M, M, K, K), mu = αβ/(α+β) (M, M, K, K), AB2 = |A-B|² (B, M, M, 1, 1),
    P (B, M, M, K, K, 3) the product centre, and K_ab = exp(-mu AB2) (B, M, M, K, K).
    """
    a = alpha[:, None, :, None]  # (M, 1, K, 1)
    b = alpha[None, :, None, :]  # (1, M, 1, K)
    zeta, mu = a + b, a * b / (a + b)
    d = centres[:, :, None, :] - centres[:, None, :, :]  # (B, M, M, 3)
    AB2 = torch.einsum("bmna,bmna->bmn", d, d)[..., None, None]  # a summed
    A = centres[:, :, None, None, None, :]  # (B, M, 1, 1, 1, 3)
    B = centres[:, None, :, None, None, :]  # (B, 1, M, 1, 1, 3)
    P = (a[..., None] * A + b[..., None] * B) / zeta[..., None]
    return {"zeta": zeta, "mu": mu, "AB2": AB2, "P": P, "K_ab": torch.exp(-mu * AB2)}


def contract2(coef: torch.Tensor, prim: torch.Tensor) -> torch.Tensor:
    """X_bμν = c_μp c_νq X_bμνpq   (p, q summed)."""
    return torch.einsum("mp,nq,bmnpq->bmn", coef, coef, prim)


def overlap(pairs: dict, coef: torch.Tensor) -> torch.Tensor:
    """S_μν, (B, M, M)."""
    prim = (math.pi / pairs["zeta"]) ** 1.5 * pairs["K_ab"]
    return contract2(coef, prim)


def kinetic(pairs: dict, coef: torch.Tensor) -> torch.Tensor:
    """T_μν = ⟨φ_μ| -½∇² |φ_ν⟩, (B, M, M)."""
    s = (math.pi / pairs["zeta"]) ** 1.5 * pairs["K_ab"]
    prim = pairs["mu"] * (3 - 2 * pairs["mu"] * pairs["AB2"]) * s
    return contract2(coef, prim)


def nuclear_attraction(pairs: dict, coef: torch.Tensor, nuclei: torch.Tensor, Z: torch.Tensor) -> torch.Tensor:
    """V_μν = ⟨φ_μ| -Σ_c Z_c/|r - C_c| |φ_ν⟩.   nuclei (B, A, 3), Z (A,) → (B, M, M); c summed."""
    PC = pairs["P"][..., None, :] - nuclei[:, None, None, None, None, :, :]  # (B, M, M, K, K, A, 3)
    PC2 = torch.einsum("bmnpqca,bmnpqca->bmnpqc", PC, PC)
    zeta = pairs["zeta"][..., None]
    per_nucleus = -2 * math.pi / zeta * pairs["K_ab"][..., None] * boys_f0(zeta * PC2)
    return contract2(coef, torch.einsum("c,bmnpqc->bmnpq", Z, per_nucleus))


def electron_repulsion(pairs: dict, coef: torch.Tensor) -> torch.Tensor:
    """(μν|λσ) = ∫∫ φ_μ(1)φ_ν(1) r₁₂⁻¹ φ_λ(2)φ_σ(2), the full (B, M, M, M, M) tensor; p q r t summed.

    The bra pair (μν, pq) and ket pair (λσ, rt) are the same pair tensors broadcast against each other.
    """

    def bra(x):  # (B, M, M, K, K, ...) → (B, M, M, 1, 1, K, K, 1, 1, ...)
        return x[:, :, :, None, None, :, :, None, None]

    def ket(x):  # → (B, 1, 1, M, M, 1, 1, K, K, ...)
        return x[:, None, None, :, :, None, None, :, :]

    zeta, eta = bra(pairs["zeta"][None]), ket(pairs["zeta"][None])
    PQ = bra(pairs["P"]) - ket(pairs["P"])
    PQ2 = (PQ * PQ).sum(-1)  # a summed
    pref = 2 * math.pi**2.5 / (zeta * eta * torch.sqrt(zeta + eta))
    prim = pref * bra(pairs["K_ab"]) * ket(pairs["K_ab"]) * boys_f0(zeta * eta / (zeta + eta) * PQ2)
    return torch.einsum("mp,nq,lr,st,bmnlspqrt->bmnls", coef, coef, coef, coef, prim)


def density(C: torch.Tensor, n_occ: int) -> torch.Tensor:
    """P_bλσ = 2 C_bλa C_bσa over the occupied MOs a (each holds two electrons).   (B, M, M)"""
    occ = C[..., :n_occ]
    return 2 * torch.einsum("bla,bsa->bls", occ, occ)


def fock(H: torch.Tensor, P: torch.Tensor, eri: torch.Tensor) -> torch.Tensor:
    """F_μν = H_μν + P_λσ [(μν|λσ) - ½ (μλ|νσ)]   (λ, σ summed): Coulomb J minus half the exchange K."""
    J = torch.einsum("bls,bmnls->bmn", P, eri)
    K = torch.einsum("bls,bmlns->bmn", P, eri)
    return H + J - 0.5 * K


def electronic_energy(P: torch.Tensor, H: torch.Tensor, F: torch.Tensor) -> torch.Tensor:
    """E_el = ½ P_μν (H_μν + F_μν)   (μ, ν summed).   (B,)"""
    return 0.5 * torch.einsum("bmn,bmn->b", P, H + F)


def nuclear_repulsion(nuclei: torch.Tensor, Z: torch.Tensor) -> torch.Tensor:
    """Σ_{A<B} Z_A Z_B / R_AB for each geometry.   nuclei (B, A, 3) → (B,)"""
    d = nuclei[:, :, None, :] - nuclei[:, None, :, :]
    R = torch.sqrt(torch.einsum("bija,bija->bij", d, d))
    i, j = torch.triu_indices(len(Z), len(Z), offset=1)
    return (Z[i] * Z[j] / R[:, i, j]).sum(-1)


def basis_on_points(alpha: torch.Tensor, coef: torch.Tensor, centres: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    """φ_bμ(r_z) = c_μp exp(-α_μp |r_z - A_bμ|²)   (p summed).   r (Z, 3) → (B, Z, M)"""
    d = r[None, :, None, :] - centres[:, None, :, :]  # (B, Z, M, 3)
    d2 = torch.einsum("bzma,bzma->bzm", d, d)
    return torch.einsum("mp,bzmp->bzm", coef, torch.exp(-alpha[None, None] * d2[..., None]))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/hartree_fock_h2/tensors.py
"""Build the concrete tensors: the molecule, its STO-3G basis and the batch of bond lengths.

Atomic units throughout: lengths in bohr (a₀ = 0.529 Å), energies in hartree (Eₕ = 27.2 eV).
"""

import math

import torch

F64 = torch.float64

# STO-3G fit of a Slater 1s function with exponent ζ = 1 (the values in Szabo & Ostlund's Appendix B program).
# For another ζ the Gaussian exponents scale as α → α ζ²; the contraction coefficients d stay the same.
STO3G_ALPHA = (0.109818, 0.405771, 2.22766)
STO3G_D = (0.444635, 0.535328, 0.154329)

# name: (nuclear charges Z_A, Slater exponents ζ_A of the 1s function on each atom, electrons on each separated atom)
MOLECULES = {
    "H2": ((1.0, 1.0), (1.24, 1.24), (1, 1)),  # H₂ → H + H
    "HeH+": ((2.0, 1.0), (2.0925, 1.24), (2, 0)),  # HeH⁺ → He + H⁺
}
N_ELECTRONS = 2


def molecule(name: str, device: torch.device):
    """Z (A,) nuclear charges, zeta (M,) one 1s basis function per atom (M = A = 2), electrons per separated atom."""
    Z, zeta, n_atom = MOLECULES[name]
    return torch.tensor(Z, dtype=F64, device=device), torch.tensor(zeta, dtype=F64, device=device), n_atom


def sto3g(zeta: torch.Tensor):
    """Contracted Gaussians φ_μ(r) = Σ_p c_μp exp(-α_μp |r - A_μ|²).

    alpha (M, K) exponents α_μp = α_p ζ_μ², coef (M, K) = d_p (2α_μp/π)^{3/4}, the primitive normalisation
    folded into the contraction coefficient. K = 3 primitives per function.
    """
    a1 = torch.tensor(STO3G_ALPHA, dtype=F64, device=zeta.device)
    d = torch.tensor(STO3G_D, dtype=F64, device=zeta.device)
    alpha = a1[None, :] * zeta[:, None] ** 2
    coef = d[None, :] * (2 * alpha / math.pi) ** 0.75
    return alpha, coef


def bond_lengths(r_min: float, r_max: float, n: int, device: torch.device) -> torch.Tensor:
    """R_b, shape (B,): the bond lengths of the scan, in bohr."""
    return torch.linspace(r_min, r_max, n, dtype=F64, device=device)


def geometry(R: torch.Tensor) -> torch.Tensor:
    """Nuclear positions (B, A, 3): atom 0 at the origin, atom 1 at (0, 0, R_b). The basis sits on the nuclei."""
    pos = torch.zeros(len(R), 2, 3, dtype=F64, device=R.device)
    pos[:, 1, 2] = R
    return pos


def axis_points(z_min: float, z_max: float, n: int, device: torch.device) -> torch.Tensor:
    """Points (Z, 3) on the bond axis, for drawing the orbitals."""
    pts = torch.zeros(n, 3, dtype=F64, device=device)
    pts[:, 2] = torch.linspace(z_min, z_max, n, dtype=F64, device=device)
    return pts


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/hartree_fock_h2/solve.py
"""Integrals for every bond length, the batched SCF loop, a minimal-basis full CI and the separated atoms."""

import math

import torch

from problems._common.progress import steps

from . import equation as eq


def integrals(alpha: torch.Tensor, coef: torch.Tensor, nuclei: torch.Tensor, Z: torch.Tensor) -> dict:
    """S, T, V, H = T + V (B, M, M) and the two-electron tensor eri (B, M, M, M, M) for every geometry."""
    pairs = eq.gaussian_pairs(alpha, nuclei)  # the basis functions sit on the nuclei
    S = eq.overlap(pairs, coef)
    T = eq.kinetic(pairs, coef)
    V = eq.nuclear_attraction(pairs, coef, nuclei, Z)
    return {"S": S, "T": T, "V": V, "H": T + V, "eri": eq.electron_repulsion(pairs, coef)}


def symmetric_orthogonaliser(S: torch.Tensor) -> torch.Tensor:
    """X = S^{-1/2} = U s^{-1/2} Uᵀ, so that Xᵀ S X = 1.   (B, M, M)"""
    s, U = torch.linalg.eigh(S)
    return U @ torch.diag_embed(s.rsqrt()) @ U.transpose(-1, -2)


def scf(ints: dict, n_electrons: int, tol: float = 1e-10, max_iter: int = 100) -> dict:
    """Roothaan–Hall SCF for every bond length at once, starting from P = 0 (the core-Hamiltonian guess).

    Each pass: F(P) → F' = Xᵀ F X → eigh → C = X C' → new P. Stops when max |ΔP| < tol for all geometries.
    """
    H, eri = ints["H"], ints["eri"]
    X = symmetric_orthogonaliser(ints["S"])
    P = torch.zeros_like(H)
    n_occ = n_electrons // 2
    history = []
    for it in steps(max_iter, "SCF iterations"):
        F = eq.fock(H, P, eri)
        eps, C_prime = torch.linalg.eigh(X.transpose(-1, -2) @ F @ X)  # orbital energies ascending
        C = X @ C_prime  # (B, M, M): column a is MO a in the basis φ_μ
        P_new = eq.density(C, n_occ)
        change = (P_new - P).abs().amax().item()
        P = P_new
        history.append(change)
        n_iter = it + 1
        if change < tol:
            break
    F = eq.fock(H, P, eri)  # Fock matrix of the converged density
    C = _fix_sign(C)
    return {
        "P": P,
        "F": F,
        "C": C,
        "eps": eps,
        "E_el": eq.electronic_energy(P, H, F),
        "n_iter": n_iter,
        "converged": change < tol,
        "history": history,
    }


def _fix_sign(C: torch.Tensor) -> torch.Tensor:
    """An eigenvector's overall sign is arbitrary: make the coefficient on atom 1 positive, for smooth plots."""
    sign = torch.sign(C[:, 1:2, :])
    return C * torch.where(sign == 0, torch.ones_like(sign), sign)


def full_ci(ints: dict, C: torch.Tensor) -> torch.Tensor:
    """Exact (full CI) ground state of 2 electrons in the 2 MOs: lowest eigenvalue of a 3 × 3 singlet matrix.

    Configurations |1²⟩ (RHF), |1¹2¹⟩ singlet, |2²⟩. MO integrals h_ij = C_μi H_μν C_νj and
    (ij|kl) = C_μi C_νj C_λk C_σl (μν|λσ) (μ ν λ σ summed). Returns electronic energies (B,).
    """
    h = torch.einsum("bmi,bmn,bnj->bij", C, ints["H"], C)
    g = torch.einsum("bmi,bnj,bpk,bql,bmnpq->bijkl", C, C, C, C, ints["eri"])
    r2 = math.sqrt(2.0)
    H00 = 2 * h[:, 0, 0] + g[:, 0, 0, 0, 0]
    H11 = h[:, 0, 0] + h[:, 1, 1] + g[:, 0, 0, 1, 1] + g[:, 0, 1, 1, 0]
    H22 = 2 * h[:, 1, 1] + g[:, 1, 1, 1, 1]
    H01 = r2 * (h[:, 0, 1] + g[:, 0, 0, 0, 1])  # = √2 F₁₂ = 0 at convergence (Brillouin's theorem)
    H12 = r2 * (h[:, 0, 1] + g[:, 0, 1, 1, 1])
    H02 = g[:, 0, 1, 0, 1]  # exchange integral K₁₂ couples |1²⟩ and |2²⟩
    M = torch.stack(
        [torch.stack([H00, H01, H02], -1), torch.stack([H01, H11, H12], -1), torch.stack([H02, H12, H22], -1)], -2
    )
    return torch.linalg.eigvalsh(M)[:, 0]


def separated_atoms(alpha: torch.Tensor, coef: torch.Tensor, Z: torch.Tensor, n_atom: tuple) -> float:
    """Energy of the atoms far apart, each with its own 1s function: one electron gives h, two give 2h + (11|11)."""
    total = 0.0
    for k, n in enumerate(n_atom):
        nucleus = torch.zeros(1, 1, 3, dtype=alpha.dtype, device=alpha.device)
        ints = integrals(alpha[k : k + 1], coef[k : k + 1], nucleus, Z[k : k + 1])
        h, J = ints["H"][0, 0, 0].item(), ints["eri"][0, 0, 0, 0, 0].item()
        total += n * h + (J if n == 2 else 0.0)
    return total


def equilibrium(R: torch.Tensor, E: torch.Tensor) -> tuple[float, float]:
    """Bond length and energy at the minimum: a parabola through the lowest grid point and its neighbours."""
    k = int(torch.argmin(E).clamp(1, len(E) - 2))
    x, y = R[k - 1 : k + 2], E[k - 1 : k + 2]
    A = torch.stack([x**2, x, torch.ones_like(x)], -1)
    a, b, c = torch.linalg.solve(A, y).tolist()
    r0 = -b / (2 * a)
    return r0, a * r0**2 + b * r0 + c


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/hartree_fock_h2/plot.py
"""Animate the bond-length scan: potential curves, the two MOs along the axis, and the orbital energies."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif

BOND, ANTI = "#2563eb", "#dc2626"


def animate(
    name: str,
    R: torch.Tensor,
    E: torch.Tensor,
    E_ci: torch.Tensor,
    E_atoms: float,
    eps: torch.Tensor,
    C: torch.Tensor,
    z: torch.Tensor,
    psi: torch.Tensor,
    eq_point: tuple[float, float],
    path: Path,
) -> Path:
    """R, E, E_ci (B,); eps (B, 2); C (B, 2, 2); z (Z,) positions on the axis; psi (B, Z, 2) MOs along it."""
    R, E, E_ci, eps, C = (x.cpu().numpy() for x in (R, E, E_ci, eps, C))
    z, psi = z.cpu().numpy(), psi.cpu().numpy()
    atoms = ("H", "H") if name == "H2" else ("He", "H")
    limit = "H + H" if name == "H2" else "He + H⁺"
    pretty = "H₂" if name == "H2" else "HeH⁺"

    fig, (ax_e, ax_mo, ax_eps) = plt.subplots(1, 3, figsize=(10.5, 3.9), gridspec_kw={"width_ratios": [1.1, 1.2, 0.9]})
    lo, hi = min(E.min(), E_ci.min()), max(E.max(), E_ci.max(), E_atoms)
    ax_e.plot(R, E, color="#d1d5db", lw=1)
    ax_e.plot(R, E_ci, color="#d1d5db", lw=1, ls="--")
    (line_hf,) = ax_e.plot([], [], color=BOND, lw=2, label="RHF")
    (line_ci,) = ax_e.plot([], [], color="#16a34a", lw=1.5, ls="--", label="full CI (same basis)")
    ax_e.axhline(E_atoms, color="#f59e0b", lw=1, ls=":", label=f"{limit} limit")
    ax_e.plot(*eq_point, "o", color="#374151", ms=4)
    ax_e.annotate(f"R_e = {eq_point[0]:.3f}", eq_point, xytext=(8, -14), textcoords="offset points", fontsize=8)
    (dot,) = ax_e.plot([], [], "o", color=BOND, ms=6)
    ax_e.set_xlim(R[0], R[-1])
    ax_e.set_ylim(lo - 0.05, min(hi, lo + 1.6) + 0.05)
    ax_e.set_xlabel("bond length R (bohr)")
    ax_e.set_ylabel("total energy (hartree)")
    ax_e.set_title(f"{pretty} potential curve", fontsize=10)
    ax_e.legend(fontsize=7, loc="upper right")

    amp = 1.1 * abs(psi).max()
    (mo_b,) = ax_mo.plot([], [], color=BOND, lw=2, label="ψ₁ bonding (occupied)")
    (mo_a,) = ax_mo.plot([], [], color=ANTI, lw=2, label="ψ₂ antibonding")
    nuc = ax_mo.scatter([0, 0], [0, 0], s=[60, 40], c="#374151", zorder=3)
    labels = [ax_mo.text(0, -0.92 * amp, a, ha="center", fontsize=9) for a in atoms]
    ax_mo.axhline(0, color="#9ca3af", lw=0.8)
    coef_text = ax_mo.text(0.02, 0.97, "", transform=ax_mo.transAxes, va="top", fontsize=8, family="monospace")
    ax_mo.set_xlim(z[0], z[-1])
    ax_mo.set_ylim(-amp, amp)
    ax_mo.set_xlabel("position along the bond z (bohr)")
    ax_mo.set_ylabel("ψ(z)  (bohr$^{-3/2}$)")
    ax_mo.legend(fontsize=7, loc="upper right")
    title_mo = ax_mo.set_title("", fontsize=10)

    ax_eps.plot(R, eps[:, 0], color="#d1d5db", lw=1)
    ax_eps.plot(R, eps[:, 1], color="#d1d5db", lw=1)
    (e1,) = ax_eps.plot([], [], color=BOND, lw=2, label="ε₁")
    (e2,) = ax_eps.plot([], [], color=ANTI, lw=2, label="ε₂")
    ax_eps.axhline(0, color="#9ca3af", lw=0.8)
    ax_eps.set_xlim(R[0], R[-1])
    ax_eps.set_xlabel("bond length R (bohr)")
    ax_eps.set_ylabel("orbital energy (hartree)")
    ax_eps.set_title("Fock eigenvalues", fontsize=10)
    ax_eps.legend(fontsize=7)

    def update(b: int):
        line_hf.set_data(R[: b + 1], E[: b + 1])
        line_ci.set_data(R[: b + 1], E_ci[: b + 1])
        dot.set_data([R[b]], [E[b]])
        mo_b.set_data(z, psi[b, :, 0])
        mo_a.set_data(z, psi[b, :, 1])
        nuc.set_offsets([[0, 0], [R[b], 0]])
        labels[0].set_x(0)
        labels[1].set_x(R[b])
        coef_text.set_text(
            f"C = [{C[b, 0, 0]:+.3f} {C[b, 0, 1]:+.3f}]\n    [{C[b, 1, 0]:+.3f} {C[b, 1, 1]:+.3f}]\nE = {E[b]:.4f} Eh"
        )
        title_mo.set_text(f"molecular orbitals at R = {R[b]:.2f} bohr")
        e1.set_data(R[: b + 1], eps[: b + 1, 0])
        e2.set_data(R[: b + 1], eps[: b + 1, 1])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(R), 60), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'molecule': 'H2',
    'R_min': 0.6,
    'R_max': 6.0,
    'n_bond_lengths': 100
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.hartree_fock_h2")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.hartree_fock_h2.main")
problem_dir = Path("problems/physchem/hartree_fock_h2")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
